Q1


In [13]:
from pyspark.sql import SparkSession
import numpy as np

# Initialize SparkSession
spark = SparkSession.builder.appName("TIPS").getOrCreate()

In [19]:
#a)

data=spark.read.csv("tips.csv", header=True, inferSchema=True)
data.printSchema()
data.show(10)

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)
 |-- price_per_person: double (nullable = true)
 |-- Payer Name: string (nullable = true)
 |-- CC Number: double (nullable = true)
 |-- Payment ID: string (nullable = true)

+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer Name| CC Number|Payment ID|
+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|3.56033E15|   Sun2959|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker|4.47807E15|   Sun4608|
|     21.01| 3.5|  Male|    No

In [23]:
from pyspark.sql.functions import round

#b)
percentage_data=data.withColumn("Tip_Percentage",round((data.tip/data.total_bill)*100,2))

In [24]:
percentage_data.show()

+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+--------------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer Name| CC Number|Payment ID|Tip_Percentage|
+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+--------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|3.56033E15|   Sun2959|          5.94|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker|4.47807E15|   Sun4608|         16.05|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|             7.0|    Travis Walters|6.01181E15|   Sun4458|         16.66|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|           11.84|  Nathaniel Harris|4.67614E15|   Sun5260|         13.98|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|            6.15|      Tonya Carter|4.83273E15|   Sun2251|         14.68|
|     25.29|4.71|  Male|    No|S

In [30]:
#c)
percentage_data.createOrReplaceTempView("tips")

In [33]:
total_average = spark.sql("SELECT day,AVG(total_bill) AS Average_total_bill FROM tips GROUP BY day")
total_average.show()

+----+------------------+
| day|Average_total_bill|
+----+------------------+
|Thur|17.682741935483865|
| Sun|21.410000000000004|
| Sat|20.441379310344825|
| Fri|17.151578947368417|
+----+------------------+



In [34]:
max_tip=spark.sql("SELECT gender,MAX(tip) FROM tips GROUP BY gender")
max_tip.show()

+------+--------+
|gender|max(tip)|
+------+--------+
|Female|     6.5|
|  Male|    10.0|
+------+--------+



In [35]:
tip_percentage=spark.sql("SELECT * FROM tips where Tip_Percentage>20")
tip_percentage.show()

+----------+----+------+------+----+------+----+----------------+------------------+----------+----------+--------------+
|total_bill| tip|gender|smoker| day|  time|size|price_per_person|        Payer Name| CC Number|Payment ID|Tip_Percentage|
+----------+----+------+------+----+------+----+----------------+------------------+----------+----------+--------------+
|      8.77| 2.0|  Male|    No| Sun|Dinner|   2|            4.38|Kristopher Johnson|2.22373E15|   Sun5985|         22.81|
|     14.78|3.23|  Male|    No| Sun|Dinner|   2|            7.39|     Jerome Abbott|3.53212E15|   Sun3775|         21.85|
|     14.83|3.02|Female|    No| Sun|Dinner|   2|            7.42|     Vanessa Jones|3.00167E13|   Sun3848|         20.36|
|     16.29|3.71|  Male|    No| Sun|Dinner|   3|            5.43|      John Pittman|6.52134E15|   Sun2998|         22.77|
|     16.97| 3.5|Female|    No| Sun|Dinner|   3|            5.66|    Laura Martinez|3.04223E13|   Sun2789|         20.62|
|     17.92|4.08|  Male|

In [36]:
percentage_data.write.parquet("tips.parquet")